# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saibalaji2004/Flyrank-intern-repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and rationale

This project addresses a prioritization problem: identifying pages that should be reviewed first when their search performance may be declining. I will begin with Logistic Regression because it provides a relatively interpretable classification baseline and can produce probabilities for ranking pages by their estimated likelihood of the observed decline label.

The target label is derived from `trend_direction`, where `down` indicates an observed decline. This is a proxy outcome, not a confirmed content-quality problem. The model is intended to support human review, not prove causation or predict Google's ranking algorithm.

I will compare the trained model against the existing Week-4 rule-based baseline using the same evaluation split and ranking metrics. Model complexity will only be increased if the results justify it.

In [8]:
# ML-08: Initial data and target checks
import pandas as pd
import numpy as np

dataset_url = (
    "https://raw.githubusercontent.com/"
    "saibalaji2004/Flyrank-intern-repo/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(dataset_url)

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

# The target is an outcome, never a predictive feature.
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts(dropna=False))
print("\nTarget rate:")
print(df["is_declining_label"].mean())

# Leakage checks
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
}

print("\nForbidden predictive fields:", sorted(forbidden_features))
assert "trend_direction" not in forbidden_features - {"trend_direction"}
assert df["content_id"].is_unique

Dataset shape: (30000, 44)
Unique content IDs: 30000
Unique clients: 32

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

Forbidden predictive fields: ['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design and rationale

I will use a grouped train/test split based on `client_id`, with 80% of clients assigned to training and 20% to testing. Grouping prevents pages from the same client from appearing in both sets, reducing the risk that client-specific patterns inflate evaluation results.

The split will use a fixed random seed for reproducibility. The target label is `is_declining_label`, derived from `trend_direction == "down"`. The fields `trend_direction`, `trend_pct`, and `is_declining_label` will not be used as predictive features. The pseudonymous identifiers `content_id` and `client_id` will also be excluded from the feature matrix.

This is a client-held-out evaluation, not a time-based forecast. Results will therefore describe performance on held-out clients in this dataset and will not guarantee performance on future search data.

In [9]:
# ML-08: Client-grouped train/test split
from sklearn.model_selection import GroupShuffleSplit

TARGET = "is_declining_label"
GROUP = "client_id"
RANDOM_STATE = 42

# Keep only rows with a known group and target.
model_df = df.dropna(subset=[GROUP, TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        y=model_df[TARGET],
        groups=model_df[GROUP],
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df[GROUP].unique())
test_clients = set(test_df[GROUP].unique())

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Training decline rate:", round(train_df[TARGET].mean(), 4))
print("Testing decline rate:", round(test_df[TARGET].mean(), 4))
print("Random seed:", RANDOM_STATE)

# Critical check: no client may occur in both sets.
assert train_clients.isdisjoint(test_clients)
assert len(train_df) + len(test_df) == len(model_df)

print("\nPASS: train and test clients do not overlap.")

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Training decline rate: 0.5501
Testing decline rate: 0.511
Random seed: 42

PASS: train and test clients do not overlap.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model training and baseline comparison

I will begin with Logistic Regression as an interpretable classifier for the observed decline label. The model will be trained only on the training clients, using predictors that exclude identifiers and all outcome-derived fields.

I will reproduce the existing Week-4 baseline scoring rules without modifying the original baseline notebook. Both approaches will be evaluated on the same held-out test clients using Precision@10, Precision@20, and the test-set decline rate as the reference base rate.

The model will rank pages by their estimated probability of decline. Preprocessing will be fitted on training data only, and the random seed will be fixed for reproducibility. The comparison will be treated as decision-support evidence for this dataset, not as proof of future performance.

In [10]:

# ML-08: Train Logistic Regression and compare with Week-4 baseline

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

# Reproducibility
RANDOM_STATE = 42
TARGET = "is_declining_label"

# Fields that must never be predictive features
identifier_and_target_fields = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

# These compare the latest 30 days with the previous 30 days.
# Exclude them to reduce the risk of revealing the target construction.
recent_comparison_fields = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

excluded_fields = identifier_and_target_fields | recent_comparison_fields

# Use the conservative, preselected feature set.
# Do not use identifiers, target fields, or direct 30-day comparison fields.
feature_columns = [
    col for col in df.columns
    if col not in excluded_fields
]

X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()
y_train = train_df[TARGET].astype(int)
y_test = test_df[TARGET].astype(int)

# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in feature_columns
    if col not in numeric_features
]

# Fit all preprocessing using training data only
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    )),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=RANDOM_STATE
    )),
])

model.fit(X_train, y_train)

# Rank held-out pages by predicted probability of decline
model_scores = model.predict_proba(X_test)[:, 1]

# Reproduce the Week-4 rule-based baseline.
# Keep its existing threshold of 731 impressions and its score weights.
baseline_stale = (
    test_df["days_since_last_update"].ge(90)
).fillna(False)

baseline_visible = (
    test_df["impressions_90d"].ge(731)
).fillna(False)

baseline_scores = (
    2 * baseline_stale.astype(int)
    + baseline_visible.astype(int)
)

# Precision@K for pages ranked by each approach
def precision_at_k(y_true, scores, k):
    ranking = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(y_true)[ranking].mean())

results = pd.DataFrame([
    {
        "Approach": "Week-4 rule-based baseline",
        "Precision@10": precision_at_k(
            y_test, baseline_scores, 10
        ),
        "Precision@20": precision_at_k(
            y_test, baseline_scores, 20
        ),
    },
    {
        "Approach": "Logistic Regression",
        "Precision@10": precision_at_k(
            y_test, model_scores, 10
        ),
        "Precision@20": precision_at_k(
            y_test, model_scores, 20
        ),
    },
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training features:", len(feature_columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Training decline rate:", round(y_train.mean(), 4))
print("Held-out test decline rate:", round(y_test.mean(), 4))

print("\nBaseline vs model on the same held-out clients:")
display(results.round(3))

print("\nAdditional model diagnostics:")
print("Average precision:", round(
    average_precision_score(y_test, model_scores), 4
))
print("ROC-AUC:", round(
    roc_auc_score(y_test, model_scores), 4
))

# Leakage and split safeguards
assert not (set(feature_columns) & excluded_fields)
assert set(train_df["client_id"]).isdisjoint(
    set(test_df["client_id"])
)
assert len(model_scores) == len(test_df)

print("\nPASS: excluded fields are not model features.")
print("PASS: training and test clients do not overlap.")


Training rows: 23837
Testing rows: 6163
Training features: 34
Numeric features: 23
Categorical features: 11
Training decline rate: 0.5501
Held-out test decline rate: 0.511

Baseline vs model on the same held-out clients:


,Approach,Precision@10,Precision@20
0,Week-4 rule-based baseline,0.4,0.40
1,Logistic Regression,0.6,0.65



Additional model diagnostics:
Average precision: 0.564
ROC-AUC: 0.5726

PASS: excluded fields are not model features.
PASS: training and test clients do not overlap.


### Results: Logistic Regression vs. the Week-4 baseline

The dataset was split by client, with 23,837 rows from 25 clients used for training and 6,163 rows from 7 separate clients used for testing. The training decline-label rate was 55.01%, while the held-out test rate was 51.10%.

On the held-out test set, the Week-4 rule-based baseline achieved Precision@10 of 0.40 and Precision@20 of 0.40. Logistic Regression achieved Precision@10 of 0.60 and Precision@20 of 0.65.

At the top 10, Logistic Regression identified 6 pages carrying the observed decline label, compared with 4 for the baseline. At the top 20, it identified 13 such pages, compared with 8 for the baseline. This represents an absolute improvement of 20 percentage points at Precision@10 and 25 percentage points at Precision@20.

The model's average precision was 0.564 and its ROC-AUC was 0.573. These overall ranking metrics indicate limited discrimination, despite the improvement among the highest-ranked pages. The results suggest that Logistic Regression may be useful for prioritizing a small review queue, but its performance is not strong enough to justify automated decisions.

The evaluation uses held-out clients from the same anonymized dataset, not a future time period. The label is an observed trend proxy, not proof that a page has poor content quality or that refreshing it will improve search performance. Further validation is required before operational use.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error analysis and interpretation

At a classification threshold of 0.5, Logistic Regression achieved 56% accuracy on the held-out test set. Its precision for the observed decline label was 0.55, with recall of 0.74 and an F1-score of 0.63. For pages not labelled as declining, precision was 0.57 and recall was 0.37.

The confusion matrix contained 2,321 true positives, 1,103 true negatives, 1,911 false positives, and 828 false negatives. The model identified many pages labelled as declining, but it also incorrectly flagged a substantial number of pages without that label. Its recall for the decline label was higher than its precision, indicating that it captured many declining-labelled pages at the cost of additional false alarms.

The mean predicted probability was 0.583 for pages labelled as declining and 0.532 for pages not labelled as declining. This separation is modest and is consistent with the ROC-AUC of 0.573. The model therefore shows limited overall discrimination, even though it performed better than the rule-based baseline at Precision@10 and Precision@20.

These errors matter operationally. False positives can waste content-team review time, while false negatives can leave pages with declining observed performance out of the review queue. The model should therefore be treated as a prioritization aid for human review, not as an automatic decision-maker. The label represents an observed trend and does not establish why a page declined.

In [11]:

# ML-08: Error analysis on held-out test clients

from sklearn.metrics import confusion_matrix, classification_report

# Use 0.5 as a diagnostic threshold
test_predictions = (model_scores >= 0.5).astype(int)

print("Classification report at threshold 0.5:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Not labelled declining", "Labelled declining"],
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions))

# Compare predicted probabilities by actual label
error_summary = pd.DataFrame({
    "actual_label": np.asarray(y_test),
    "predicted_probability": model_scores
})

print("\nPredicted probability summary by actual label:")
display(
    error_summary.groupby("actual_label")["predicted_probability"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
)

# Categorize incorrect predictions
error_summary["prediction"] = test_predictions

error_summary["error_type"] = np.select(
    [
        (error_summary["actual_label"] == 0) &
        (error_summary["prediction"] == 1),
        (error_summary["actual_label"] == 1) &
        (error_summary["prediction"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct"
)

print("\nError counts:")
print(error_summary["error_type"].value_counts())


Classification report at threshold 0.5:
                        precision    recall  f1-score   support

Not labelled declining       0.57      0.37      0.45      3014
    Labelled declining       0.55      0.74      0.63      3149

              accuracy                           0.56      6163
             macro avg       0.56      0.55      0.54      6163
          weighted avg       0.56      0.56      0.54      6163


Confusion matrix:
[[1103 1911]
 [ 828 2321]]

Predicted probability summary by actual label:


,count,mean,median,min,max
actual_label,,,,,
0,3014,0.532,0.567,0.024,0.921
1,3149,0.583,0.615,0.038,0.940



Error counts:
error_type
Correct           3424
False positive    1911
False negative     828
Name: count, dtype: int64


In [12]:

# ML-08: Feature importance and representative errors

# Extract coefficients from the fitted Logistic Regression pipeline
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_classifier = model.named_steps["classifier"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_classifier.coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
}).sort_values("absolute_coefficient", ascending=False)

print("Top 10 features by absolute Logistic Regression coefficient:")
display(
    importance[["feature", "coefficient"]]
    .head(10)
    .round(4)
)

print(
    "\nInterpretation: positive coefficients increase the model's "
    "estimated odds of the decline label; negative coefficients "
    "decrease them, holding other encoded features constant."
)

# Build an audit table in exactly the same order as model predictions
audit = X_test.reset_index(drop=True).copy()
audit["actual_label"] = np.asarray(y_test)
audit["predicted_probability"] = model_scores
audit["predicted_label"] = test_predictions

audit["error_type"] = np.select(
    [
        (audit["actual_label"] == 0) &
        (audit["predicted_label"] == 1),
        (audit["actual_label"] == 1) &
        (audit["predicted_label"] == 0),
    ],
    ["False positive", "False negative"],
    default="Correct"
)

# Display only a small, safe set of useful fields
display_features = [
    col for col in [
        "content_type",
        "main_intent",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "ctr",
    ]
    if col in audit.columns
]

for error_type in ["False positive", "False negative"]:
    examples = audit[audit["error_type"] == error_type].copy()

    # Show the three most confident examples of each error type
    if error_type == "False positive":
        examples = examples.sort_values(
            "predicted_probability", ascending=False
        )
    else:
        examples = examples.sort_values(
            "predicted_probability", ascending=True
        )

    print(f"\nThree representative {error_type.lower()} cases:")
    display(
        examples[
            display_features
            + ["actual_label", "predicted_probability", "error_type"]
        ].head(3).round(3)
    )


Top 10 features by absolute Logistic Regression coefficient:


,feature,coefficient
76,categorical__position_tier_top_3,-0.9872
9,numeric__users_90d,-0.9384
8,numeric__sessions_90d,0.8319
48,categorical__model_used_gpt-5-mini,0.7947
13,numeric__days_with_impressions,0.7897
49,categorical__model_used_unknown,-0.5629
39,categorical__main_intent_navigational,-0.5230
14,numeric__days_with_sessions,-0.5086
56,categorical__freshness_tier_31-90,-0.4735
45,categorical__model_used_gemini-2.5-flash,-0.4314



Interpretation: positive coefficients increase the model's estimated odds of the decline label; negative coefficients decrease them, holding other encoded features constant.

Three representative false positive cases:


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,actual_label,predicted_probability,error_type
4309,keyword article,informational,275,104,3115,0,12.8,0.00,0,0.921,False positive
2103,keyword article,commercial,181,20,235,2,31.0,0.85,0,0.919,False positive
3836,keyword article,informational,348,104,166,1,16.1,0.60,0,0.912,False positive



Three representative false negative cases:


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,actual_label,predicted_probability,error_type
874,keyword article,informational,490,22,916,0,78.6,0.0,1,0.038,False negative
2651,keyword article,informational,502,20,643,0,76.0,0.0,1,0.039,False negative
5762,keyword article,commercial,502,22,767,0,53.4,0.0,1,0.055,False negative


### Feature importance and error patterns

The largest absolute Logistic Regression coefficients were associated with the top-three position category, users over 90 days, sessions over 90 days, model-used categories, days with impressions, and days with sessions. The `position_tier_top_3` and `users_90d` coefficients were negative, while `sessions_90d` and `days_with_impressions` had positive coefficients. These directions describe associations learned by the fitted model, holding other encoded features constant; they do not establish causation.

The false-positive examples included older keyword articles with different update histories and visibility levels. Their predicted probabilities were high even though their observed labels were not declining. These cases illustrate how the model can flag pages for review when the target label does not indicate a decline.

The false-negative examples were older keyword articles with recent updates, relatively low click counts, and poor average positions. Their predicted probabilities were low even though their observed labels indicated a decline. This suggests that some declining-labelled pages may not exhibit the combinations of signals learned by the model.

These six examples are illustrative cases, not a representative sample of all errors. The model's limited ROC-AUC of 0.573 and its false-positive count show that the feature relationships are not sufficiently reliable for automatic decisions. The ranked output should be used to guide human investigation, with additional validation needed before operational deployment.

### Ranked recommendations

1. **Use the model only to prioritize human review.** Logistic Regression achieved Precision@10 of 0.60 and Precision@20 of 0.65 on held-out clients, outperforming the rule-based baseline at both cutoffs. Content teams can use the ranking to identify a small set of pages for investigation, rather than automatically refreshing pages.

2. **Investigate pages with declining performance signals.** Review impressions, clicks, average position, search intent, and recent update history together. A decline label alone does not explain why a page's performance changed.

3. **Check high-confidence false positives before making changes.** Some pages received high predicted decline probabilities without carrying the observed decline label. Reviewing these cases can help teams understand when the model over-prioritizes pages.

4. **Investigate false negatives and weak visibility signals.** Some declining-labelled pages received very low predicted probabilities. Review whether the current features adequately represent low-visibility pages and pages that have been updated recently.

5. **Validate the ranking on future time periods.** The current evaluation holds out clients but does not hold out a future time period. A chronological evaluation would provide stronger evidence about whether the model can prioritize pages in a later period.

6. **Retain the rule-based baseline as a benchmark.** Keep comparing the model and baseline using the same test split and ranking metrics. Reassess the model if its performance fails to improve consistently across new samples.

These recommendations are provisional and based on the current anonymized dataset. They are intended to support investigation, not to guarantee improvements in search rankings or traffic.

### Limitations and honest framing

- **Proxy label:** The target label is derived from `trend_direction == "down"`. It represents an observed trend, not confirmed poor content quality or a proven need for a content refresh.
- **Limited predictive performance:** Logistic Regression achieved a ROC-AUC of 0.573 and average precision of 0.564. Although it outperformed the rule-based baseline at Precision@10 and Precision@20, overall discrimination remains limited.
- **Client-held-out evaluation:** The test set contains clients excluded from training, but it does not represent a future time period. The results cannot establish how well the model will perform on future data.
- **Potential data limitations:** Several dataset fields contain missing values. Imputation and categorical encoding were used, but missingness may still affect model performance.
- **Snapshot and feature limitations:** The available data does not establish a complete chronological sequence of page changes and subsequent outcomes. The model therefore cannot prove that a particular feature caused a decline.
- **Illustrative error analysis:** The inspected false positives and false negatives are examples, not a comprehensive analysis of every error group.
- **No causal claims:** Neither the baseline nor the model establishes that refreshing content will improve impressions, clicks, or search rankings.

The model should be treated as an experimental decision-support tool. Future evaluation should include a chronological holdout, repeated testing across clients, and review of the practical value of the ranked recommendations before deployment.

### Reproducibility

The analysis uses the anonymized FlyRank content-refresh dataset available in the project repository. The notebook loads the dataset from its public repository URL, creates the decline label from `trend_direction == "down"`, and uses a grouped train/test split based on `client_id`.

The split uses random seed 42, with 80% of clients assigned to training and 20% to testing. Logistic Regression preprocessing is fitted on training data only. The model is evaluated on held-out clients using Precision@10, Precision@20, average precision, and ROC-AUC.

The Week-4 rule-based baseline is reproduced in this notebook for a direct comparison on the same test clients. Its original scoring notebook remains unchanged. To reproduce the analysis, run the notebook cells from top to bottom in a Python environment with pandas, NumPy, and scikit-learn installed.

Repository: https://github.com/saibalaji2004/Flyrank-intern-repo

Dataset source: https://github.com/saibalaji2004/Flyrank-intern-repo/blob/main/data/raw/content_refresh_anonymized.csv

This analysis was developed as part of the FlyRank ML Internship.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.